# Notebook I: 3D Medical Image Data Loading, Manipulation & Metadata

A hands-on guide for ML engineers entering medical imaging.

**What you'll learn:**
- How PET, SPECT, and CT physically work (and what their images represent)
- Loading DICOM and NIfTI files with PyDICOM and NiBabel
- 3D coordinate systems (LPS vs RAS) and affine matrices
- DICOM to NIfTI conversion
- Interactive 3D visualization

**Prerequisites:** Basic Python, NumPy, and matplotlib. No medical physics background needed.

**Data:** This notebook generates synthetic data so it runs anywhere without clinical data downloads. The synthetic volumes mimic the statistical and geometric properties of real PET scans.

In [ ]:
# =============================================================================
# Cell 2: Imports and Setup
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import pydicom
from pydicom.dataset import Dataset, FileDataset
from pydicom.uid import ExplicitVRLittleEndian
import pydicom.uid
import nibabel as nib
import SimpleITK as sitk
import tempfile
import os
import glob
import warnings

# Optional: ipywidgets for interactive viewer (Cell 14)
try:
    from ipywidgets import interact, IntSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not installed. Interactive viewer will use static fallback.")

warnings.filterwarnings('ignore', category=UserWarning)

# Reproducibility
np.random.seed(42)

# Matplotlib defaults for readability
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 11,
    'figure.dpi': 100,
})

print(f"pydicom version:   {pydicom.__version__}")
print(f"nibabel version:   {nib.__version__}")
print(f"SimpleITK version: {sitk.Version.VersionString()}")
print(f"NumPy version:     {np.__version__}")
print("\nAll imports successful. Ready to go!")

---
## 1. PET / SPECT / CT Physics Primer

Before touching code, you need a mental model of what these scanners measure. Each modality produces a 3D array of numbers, but those numbers mean very different things.

### 1.1 Positron Emission Tomography (PET)

**How it works:**
1. A radiotracer (most commonly **F-18 FDG**, a glucose analog) is injected into the patient.
2. FDG accumulates in metabolically active cells (tumors, brain, heart muscle).
3. F-18 decays by emitting a **positron** (the antimatter counterpart of an electron).
4. The positron travels ~1 mm before annihilating with a nearby electron.
5. Annihilation produces **two 511 keV gamma photons** traveling in **opposite directions** (180 degrees apart).
6. A ring of detectors surrounds the patient. When two detectors fire within a narrow time window (~5 ns), that's a **coincidence event** — the annihilation occurred somewhere along the line connecting them.
7. Millions of coincidence events are reconstructed into a 3D image.

**Output unit: SUV (Standardized Uptake Value)**
$$\text{SUV} = \frac{\text{tissue radioactivity concentration (kBq/mL)}}{\text{injected dose (kBq)} / \text{body weight (g)}}$$
- SUV = 1 means the tracer is distributed uniformly.
- SUV > 2.5 in a lung nodule is suspicious for malignancy.
- Brain cortex typically has SUV ~ 5-10 (the brain loves glucose).

**Typical resolution:** ~4-5 mm (limited by positron range and detector geometry).

### 1.2 Single Photon Emission Computed Tomography (SPECT)

**How it works:**
1. A radiotracer (commonly **Tc-99m** compounds) is injected.
2. Tc-99m emits **single** 140 keV gamma photons (no coincidence detection).
3. A **gamma camera** with a physical **collimator** (lead grid that only passes photons traveling perpendicular to the detector) rotates around the patient.
4. Multiple 2D projections at different angles are reconstructed into 3D.

**Key application: Myocardial Perfusion Imaging (MPI)**
- Tc-99m sestamibi/tetrofosmin traces blood flow to heart muscle.
- Compare rest vs. stress images: regions with reduced stress uptake but normal rest uptake = ischemia.
- This is one of the most common nuclear medicine exams (~10 million/year in the US).

**Typical resolution:** ~10-12 mm (worse than PET due to collimator).

### 1.3 Computed Tomography (CT)

**How it works:**
1. An **X-ray tube** and detector array rotate around the patient.
2. X-rays are attenuated (absorbed/scattered) differently by different tissues.
3. Multiple projection angles are reconstructed into cross-sectional images.

**Output unit: Hounsfield Units (HU)**
$$\text{HU} = 1000 \times \frac{\mu_{\text{tissue}} - \mu_{\text{water}}}{\mu_{\text{water}} - \mu_{\text{air}}}$$

| Material | HU Value |
|----------|----------|
| Air      | -1000    |
| Lung     | -500     |
| Fat      | -100     |
| Water    | 0        |
| Muscle   | +40      |
| Bone     | +400 to +1000 |
| Metal    | +3000+   |

**Typical resolution:** ~0.5-1.0 mm (much finer than PET/SPECT).

### 1.4 The Key Insight: Function vs. Anatomy

| | PET/SPECT | CT |
|---|---|---|
| **Measures** | Biological function (metabolism, perfusion) | Anatomical structure (density) |
| **Resolution** | Poor (4-12 mm) | Excellent (0.5-1 mm) |
| **Question answered** | *"What is happening here?"* | *"What is here?"* |

**Hybrid scanners** (PET/CT, SPECT/CT) acquire both in a single session, with the patient in the same position. The CT provides:
1. **Anatomical reference** for localizing functional abnormalities.
2. **Attenuation correction** — CT-derived attenuation maps correct for photon absorption in PET/SPECT reconstruction.

This fusion of function and anatomy is what makes nuclear medicine so powerful for oncology, cardiology, and neurology.

---
## 2. The DICOM Format

**DICOM** (Digital Imaging and Communications in Medicine) is the universal standard for medical images in clinical settings. Think of it as the JPEG of medicine, but far more complex.

### Key characteristics:

- **One file per 2D slice** (or frame). A CT scan with 512 slices = 512 files.
- **Rich metadata** embedded in every file — patient info, scanner settings, geometry.
- **Tag-based format**: Every piece of data is addressed by a `(group, element)` pair.

### Essential DICOM Tags for 3D Reconstruction

| Tag | Name | What It Means |
|-----|------|---------------|
| `(0028, 0010)` | **Rows** | Height of the 2D slice in pixels |
| `(0028, 0011)` | **Columns** | Width of the 2D slice in pixels |
| `(0028, 0030)` | **PixelSpacing** | `[row_spacing, col_spacing]` in mm |
| `(0018, 0050)` | **SliceThickness** | Thickness of each slice in mm |
| `(0020, 0032)` | **ImagePositionPatient** | 3D coordinates (x,y,z) of the top-left corner of this slice |
| `(0020, 0037)` | **ImageOrientationPatient** | 6 direction cosines: first 3 = row direction, last 3 = column direction |
| `(0020, 1041)` | **SliceLocation** | Relative position along the scan axis (for sorting) |
| `(0028, 1052)` | **RescaleIntercept** | `stored_value * slope + intercept = real_value` |
| `(0028, 1053)` | **RescaleSlope** | See above. For CT: converts to Hounsfield Units |
| `(0008, 0060)` | **Modality** | `CT`, `PT` (PET), `NM` (SPECT), `MR`, etc. |
| `(0020, 000E)` | **SeriesInstanceUID** | Unique identifier grouping slices that belong together |

### Why not just use NumPy arrays?

Without the metadata, a 3D array is just numbers. You wouldn't know:
- How big each voxel is (1mm? 5mm? anisotropic?)
- What orientation the patient was in
- Whether voxel values are raw counts, Hounsfield Units, or SUV
- Where in physical space this volume sits

The metadata IS the image. Never throw it away.

In [ ]:
# =============================================================================
# Cell 5: Generate Synthetic DICOM Series
# =============================================================================
# We create a synthetic PET-like volume with Gaussian "hot spots" that simulate
# radiotracer uptake. This lets us demonstrate all DICOM concepts without
# needing real clinical data.

def generate_synthetic_pet_dicom(output_dir, shape=(32, 64, 64)):
    """
    Generate a synthetic PET DICOM series with Gaussian blob uptake regions.
    
    Parameters
    ----------
    output_dir : str
        Directory to write DICOM files into.
    shape : tuple of int
        (n_slices, rows, cols) of the output volume.
    
    Returns
    -------
    volume : np.ndarray
        The 3D volume that was written (float32).
    """
    n_slices, rows, cols = shape
    
    # --- Create 3D volume ---
    # Background: Poisson noise simulates the stochastic nature of radioactive decay.
    # Real PET data is inherently noisy because each voxel value comes from
    # counting a finite number of annihilation events.
    volume = np.random.poisson(100, shape).astype(np.float32)
    
    # Add several "hot spots" simulating focal radiotracer uptake.
    # In a real PET scan, these might represent tumors, inflamed tissue,
    # or normal high-uptake organs (brain, heart, bladder).
    centers = [
        (16, 32, 32),   # Center of volume — e.g., a primary tumor
        (10, 20, 40),   # Upper-right region — e.g., lymph node metastasis
        (22, 45, 25),   # Lower-left — e.g., another lesion
        (16, 40, 45),   # Center-right — e.g., physiological uptake
    ]
    intensities = [800, 500, 600, 400]  # Peak SUV-like values
    sigmas = [4, 3, 5, 3]               # Spread in voxels (larger = more diffuse)
    
    # Create coordinate grids for vectorized Gaussian computation
    z, y, x = np.mgrid[0:n_slices, 0:rows, 0:cols]
    for (cz, cy, cx), intensity, sigma in zip(centers, intensities, sigmas):
        blob = intensity * np.exp(
            -((z - cz)**2 + (y - cy)**2 + (x - cx)**2) / (2 * sigma**2)
        )
        volume += blob
    
    # --- Write individual DICOM files ---
    # Each slice gets its own file, just like real DICOM series.
    # All slices share the same SeriesInstanceUID to mark them as one volume.
    series_uid = pydicom.uid.generate_uid()
    study_uid = pydicom.uid.generate_uid()
    frame_of_ref_uid = pydicom.uid.generate_uid()
    
    os.makedirs(output_dir, exist_ok=True)
    
    for i in range(n_slices):
        filename = os.path.join(output_dir, f'slice_{i:04d}.dcm')
        
        # File meta info (required DICOM preamble)
        file_meta = pydicom.Dataset()
        file_meta.MediaStorageSOPClassUID = '1.2.840.10008.5.1.4.1.1.128'  # PET Image Storage
        file_meta.MediaStorageSOPInstanceUID = pydicom.uid.generate_uid()
        file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
        
        # Create the dataset
        ds = FileDataset(filename, {}, file_meta=file_meta, preamble=b"\x00" * 128)
        
        # --- Patient / Study / Series identifiers ---
        ds.SOPClassUID = file_meta.MediaStorageSOPClassUID
        ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
        ds.StudyInstanceUID = study_uid
        ds.SeriesInstanceUID = series_uid
        ds.FrameOfReferenceUID = frame_of_ref_uid
        ds.Modality = 'PT'              # 'PT' = PET (not 'PET'!)
        ds.Manufacturer = 'Synthetic'
        ds.PatientName = 'Synthetic^Patient'
        ds.PatientID = 'SYNTH001'
        ds.StudyDate = '20260101'
        ds.SeriesDescription = 'Synthetic PET FDG'
        
        # --- Pixel data format ---
        ds.Rows = rows
        ds.Columns = cols
        ds.BitsAllocated = 16
        ds.BitsStored = 16
        ds.HighBit = 15
        ds.PixelRepresentation = 0      # 0 = unsigned
        ds.SamplesPerPixel = 1           # Grayscale
        ds.PhotometricInterpretation = 'MONOCHROME2'  # Higher value = brighter
        ds.RescaleSlope = 1.0
        ds.RescaleIntercept = 0.0
        
        # --- Spatial geometry (this is the crucial part) ---
        ds.PixelSpacing = [4.0, 4.0]    # [row_spacing, col_spacing] in mm
        ds.SliceThickness = 4.0          # mm
        ds.SpacingBetweenSlices = 4.0    # mm
        
        # ImageOrientationPatient: 6 direction cosines
        # [row_x, row_y, row_z, col_x, col_y, col_z]
        # Standard axial: row along X, column along Y
        ds.ImageOrientationPatient = [1, 0, 0, 0, 1, 0]
        
        # ImagePositionPatient: (x, y, z) of top-left pixel of this slice
        # We center the volume around the origin
        ds.ImagePositionPatient = [
            -(cols // 2) * 4.0,   # x: center along columns
            -(rows // 2) * 4.0,   # y: center along rows
            i * 4.0               # z: slice position
        ]
        ds.SliceLocation = float(i * 4.0)
        ds.InstanceNumber = i + 1
        
        # --- Encode pixel data ---
        pixel_data = np.clip(volume[i], 0, 65535).astype(np.uint16)
        ds.PixelData = pixel_data.tobytes()
        
        ds.save_as(filename)
    
    print(f"Generated {n_slices} DICOM files in: {output_dir}")
    print(f"Volume shape: {volume.shape}")
    print(f"Value range: [{volume.min():.1f}, {volume.max():.1f}]")
    print(f"Voxel size: 4.0 x 4.0 x 4.0 mm")
    
    return volume


# Generate the synthetic data
tmpdir = tempfile.mkdtemp(prefix='synthetic_pet_')
dicom_dir = os.path.join(tmpdir, 'dicom_series')
original_volume = generate_synthetic_pet_dicom(dicom_dir)

print(f"\nFiles written to: {dicom_dir}")
print(f"File listing: {sorted(os.listdir(dicom_dir))[:5]} ... ({len(os.listdir(dicom_dir))} total)")

In [ ]:
# =============================================================================
# Cell 6: Load DICOM Series with PyDICOM
# =============================================================================
# In practice, you receive a directory full of .dcm files.
# The critical steps: load all files, sort by position, build a 3D volume.

# Step 1: Find and load all DICOM files
dcm_files = sorted(glob.glob(os.path.join(dicom_dir, '*.dcm')))
print(f"Found {len(dcm_files)} DICOM files\n")

# Step 2: Read all slices into a list
slices = [pydicom.dcmread(f) for f in dcm_files]

# Step 3: Sort by SliceLocation (or ImagePositionPatient[2] for axial scans)
# IMPORTANT: File naming order is NOT guaranteed to match spatial order.
# Always sort by geometry, not filename.
sorted_slices = sorted(slices, key=lambda s: float(s.SliceLocation))

# Step 4: Inspect metadata from the first slice
ds = sorted_slices[0]
print("=" * 60)
print("KEY DICOM METADATA (first slice)")
print("=" * 60)
print(f"  Modality:                {ds.Modality}")
print(f"  Patient Name:            {ds.PatientName}")
print(f"  Study Date:              {ds.StudyDate}")
print(f"  Series Description:      {ds.SeriesDescription}")
print(f"  Rows x Columns:          {ds.Rows} x {ds.Columns}")
print(f"  Pixel Spacing (mm):      {ds.PixelSpacing}")
print(f"  Slice Thickness (mm):    {ds.SliceThickness}")
print(f"  Image Orientation:       {ds.ImageOrientationPatient}")
print(f"  Image Position (slice 0): {ds.ImagePositionPatient}")
print(f"  Rescale Slope:           {ds.RescaleSlope}")
print(f"  Rescale Intercept:       {ds.RescaleIntercept}")
print(f"  Series Instance UID:     {ds.SeriesInstanceUID[:40]}...")

# Step 5: Build the 3D volume from pixel arrays
volume_from_dicom = np.stack(
    [s.pixel_array.astype(np.float32) for s in sorted_slices],
    axis=0
)

# Step 6: Apply Rescale Slope and Intercept
# raw_stored_value * slope + intercept = real-world value
# For CT: this converts to Hounsfield Units
# For PET: this may convert to activity concentration or SUV
slope = float(ds.RescaleSlope)
intercept = float(ds.RescaleIntercept)
volume_from_dicom = volume_from_dicom * slope + intercept

print(f"\n{'=' * 60}")
print("RECONSTRUCTED VOLUME")
print(f"{'=' * 60}")
print(f"  Shape (slices, rows, cols): {volume_from_dicom.shape}")
print(f"  Dtype: {volume_from_dicom.dtype}")
print(f"  Value range: [{volume_from_dicom.min():.1f}, {volume_from_dicom.max():.1f}]")
print(f"  Mean: {volume_from_dicom.mean():.1f}")

# Compute physical dimensions
ps = [float(x) for x in ds.PixelSpacing]
st = float(ds.SliceThickness)
phys_size = (
    volume_from_dicom.shape[0] * st,
    volume_from_dicom.shape[1] * ps[0],
    volume_from_dicom.shape[2] * ps[1]
)
print(f"  Physical size (mm): {phys_size[0]:.0f} x {phys_size[1]:.0f} x {phys_size[2]:.0f}")

In [ ]:
# =============================================================================
# Cell 7: Visualize DICOM Volume — Orthogonal Slices with Crosshairs
# =============================================================================
# Medical images are viewed in three standard planes:
#   - Axial:    top-down view (the most common for CT/PET)
#   - Coronal:  front view (like facing the patient)
#   - Sagittal: side view (from the patient's left or right)
#
# We draw crosshairs to show where each plane intersects the others.

def plot_orthogonal_slices(volume, voxel_spacing, title='3D Volume',
                           slice_idx=None, cmap='hot', vmin=None, vmax=None):
    """
    Display axial, coronal, and sagittal views with crosshairs.
    
    Parameters
    ----------
    volume : np.ndarray
        3D array with shape (slices, rows, cols).
    voxel_spacing : tuple
        (slice_spacing, row_spacing, col_spacing) in mm.
    title : str
        Figure title.
    slice_idx : tuple or None
        (axial, coronal, sagittal) indices. Defaults to center.
    """
    nz, ny, nx = volume.shape
    sz, sy, sx = voxel_spacing
    
    # Default to center of volume
    if slice_idx is None:
        slice_idx = (nz // 2, ny // 2, nx // 2)
    iz, iy, ix = slice_idx
    
    if vmax is None:
        vmax = np.percentile(volume, 99.5)
    if vmin is None:
        vmin = np.percentile(volume, 0.5)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    crosshair_color = 'cyan'
    crosshair_alpha = 0.5
    crosshair_lw = 0.8
    
    # --- Axial view (slice along Z) ---
    ax = axes[0]
    axial = volume[iz, :, :]
    ax.imshow(axial, cmap=cmap, vmin=vmin, vmax=vmax,
              extent=[0, nx * sx, ny * sy, 0], aspect='equal')
    ax.axhline(y=iy * sy, color=crosshair_color, alpha=crosshair_alpha, lw=crosshair_lw)
    ax.axvline(x=ix * sx, color=crosshair_color, alpha=crosshair_alpha, lw=crosshair_lw)
    ax.set_title(f'Axial (z={iz})', fontsize=12)
    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Y (mm)')
    
    # --- Coronal view (slice along Y) ---
    ax = axes[1]
    coronal = volume[:, iy, :]
    ax.imshow(coronal, cmap=cmap, vmin=vmin, vmax=vmax,
              extent=[0, nx * sx, nz * sz, 0], aspect=sz/sx)
    ax.axhline(y=iz * sz, color=crosshair_color, alpha=crosshair_alpha, lw=crosshair_lw)
    ax.axvline(x=ix * sx, color=crosshair_color, alpha=crosshair_alpha, lw=crosshair_lw)
    ax.set_title(f'Coronal (y={iy})', fontsize=12)
    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Z (mm)')
    
    # --- Sagittal view (slice along X) ---
    ax = axes[2]
    sagittal = volume[:, :, ix]
    ax.imshow(sagittal, cmap=cmap, vmin=vmin, vmax=vmax,
              extent=[0, ny * sy, nz * sz, 0], aspect=sz/sy)
    ax.axhline(y=iz * sz, color=crosshair_color, alpha=crosshair_alpha, lw=crosshair_lw)
    ax.axvline(x=iy * sy, color=crosshair_color, alpha=crosshair_alpha, lw=crosshair_lw)
    ax.set_title(f'Sagittal (x={ix})', fontsize=12)
    ax.set_xlabel('Y (mm)')
    ax.set_ylabel('Z (mm)')
    
    plt.tight_layout()
    plt.show()


# Visualize at the center of the brightest hot spot
plot_orthogonal_slices(
    volume_from_dicom,
    voxel_spacing=(4.0, 4.0, 4.0),
    title='Synthetic PET Volume — Orthogonal Views',
    slice_idx=(16, 32, 32),   # Center of the primary "tumor"
    cmap='hot'
)

# Also show a montage of all axial slices
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('All 32 Axial Slices', fontsize=14, fontweight='bold')
vmax = np.percentile(volume_from_dicom, 99.5)
for i, ax in enumerate(axes.flat):
    if i < volume_from_dicom.shape[0]:
        ax.imshow(volume_from_dicom[i], cmap='hot', vmin=0, vmax=vmax)
        ax.set_title(f'z={i}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 3. The NIfTI Format

**NIfTI** (Neuroimaging Informatics Technology Initiative) is the standard format in research and ML pipelines. It was originally designed for brain imaging but is now used for all medical imaging research.

### Key differences from DICOM:

| Feature | DICOM | NIfTI |
|---------|-------|-------|
| **Files** | One per slice (hundreds of files) | Single file for entire volume |
| **Extensions** | `.dcm` | `.nii` (uncompressed) or `.nii.gz` (gzip compressed) |
| **Metadata** | Extremely rich (patient info, scanner settings) | Minimal (dimensions, voxel size, affine matrix) |
| **Coordinate system** | LPS (Left, Posterior, Superior) | RAS (Right, Anterior, Superior) |
| **Primary use** | Clinical (hospitals, PACS) | Research (publications, ML training) |
| **4D support** | Awkward (enhanced DICOM) | Native (e.g., time series fMRI) |

### NIfTI Header Essentials

- **`header.get_data_shape()`**: Volume dimensions, e.g., `(64, 64, 32)`
- **`header.get_zooms()`**: Voxel sizes in mm, e.g., `(4.0, 4.0, 4.0)`
- **`header.get_data_dtype()`**: Data type, e.g., `float32`
- **`img.affine`**: The 4x4 matrix mapping voxel indices to world coordinates (RAS)
- **`img.get_fdata()`**: Returns the volume as a NumPy array

### Why ML engineers should care

Most ML frameworks (MONAI, nnU-Net, TotalSegmentator) expect NIfTI input. Your data pipeline will almost always involve DICOM -> NIfTI conversion at some point.

In [ ]:
# =============================================================================
# Cell 9: Generate and Load NIfTI
# =============================================================================
# Let's create a NIfTI file from scratch, then load it back to inspect.

# --- Create a synthetic NIfTI volume ---
# We'll make a different volume (CT-like) so you can see both modalities.

ct_shape = (64, 64, 32)  # (rows, cols, slices) — NIfTI uses (x, y, z) ordering
ct_volume = np.zeros(ct_shape, dtype=np.float32)

# Simulate an elliptical body cross-section
y_grid, x_grid, z_grid = np.mgrid[0:ct_shape[0], 0:ct_shape[1], 0:ct_shape[2]]
cy, cx = ct_shape[0] // 2, ct_shape[1] // 2

# Body outline (soft tissue ~ +40 HU)
body_mask = ((y_grid - cy)**2 / 20**2 + (x_grid - cx)**2 / 25**2) < 1
ct_volume[body_mask] = 40.0

# Lung regions (air ~ -500 HU)
lung_left = ((y_grid - cy)**2 / 10**2 + (x_grid - (cx - 10))**2 / 8**2) < 1
lung_right = ((y_grid - cy)**2 / 10**2 + (x_grid - (cx + 10))**2 / 8**2) < 1
ct_volume[lung_left & body_mask] = -500.0
ct_volume[lung_right & body_mask] = -500.0

# Spine (bone ~ +700 HU)
spine_mask = ((y_grid - (cy + 15))**2 + (x_grid - cx)**2) < 3**2
ct_volume[spine_mask & body_mask] = 700.0

# Background air (-1000 HU)
ct_volume[~body_mask] = -1000.0

# Add some noise
ct_volume += np.random.normal(0, 10, ct_shape).astype(np.float32)

# --- Build an affine matrix ---
# RAS convention: 4mm isotropic voxels, centered at origin
voxel_size = 4.0
affine_nifti = np.diag([voxel_size, voxel_size, voxel_size, 1.0])
# Shift origin so the volume is centered
affine_nifti[:3, 3] = -np.array(ct_shape[:3]) * voxel_size / 2

# --- Save as NIfTI ---
nifti_path = os.path.join(tmpdir, 'synthetic_ct.nii.gz')
nifti_img = nib.Nifti1Image(ct_volume, affine_nifti)
nib.save(nifti_img, nifti_path)
print(f"Saved NIfTI to: {nifti_path}")
print(f"File size: {os.path.getsize(nifti_path) / 1024:.1f} KB\n")

# --- Load it back ---
loaded_img = nib.load(nifti_path)

print("=" * 60)
print("NIfTI HEADER INSPECTION")
print("=" * 60)
print(f"  Data shape:    {loaded_img.header.get_data_shape()}")
print(f"  Voxel sizes:   {loaded_img.header.get_zooms()} mm")
print(f"  Data dtype:    {loaded_img.header.get_data_dtype()}")
print(f"  Affine matrix:")
print(loaded_img.affine)

# Get the actual data
ct_data = loaded_img.get_fdata()
print(f"\n  Loaded array shape: {ct_data.shape}")
print(f"  Value range: [{ct_data.min():.1f}, {ct_data.max():.1f}] HU")

# --- Visualize the CT volume ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Synthetic CT Volume (Hounsfield Units)', fontsize=14, fontweight='bold')

# NIfTI shape is (x, y, z), so we need to pick slices along each axis
mid = [s // 2 for s in ct_data.shape]

im0 = axes[0].imshow(ct_data[:, :, mid[2]].T, cmap='gray', vmin=-1000, vmax=1000, origin='lower')
axes[0].set_title(f'Axial (z={mid[2]})')
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')

im1 = axes[1].imshow(ct_data[:, mid[1], :].T, cmap='gray', vmin=-1000, vmax=1000, origin='lower')
axes[1].set_title(f'Coronal (y={mid[1]})')
axes[1].set_xlabel('X'); axes[1].set_ylabel('Z')

im2 = axes[2].imshow(ct_data[mid[0], :, :].T, cmap='gray', vmin=-1000, vmax=1000, origin='lower')
axes[2].set_title(f'Sagittal (x={mid[0]})')
axes[2].set_xlabel('Y'); axes[2].set_ylabel('Z')

plt.colorbar(im2, ax=axes, label='HU', shrink=0.8)
plt.tight_layout()
plt.show()

---
## 4. Coordinate Systems Deep Dive

This is where most ML engineers get confused. Medical images exist in **physical space** (millimeters), not just pixel space. Two different scans of the same patient (e.g., PET and CT) can be overlaid because they share a coordinate system.

### 4.1 LPS — DICOM Standard

Imagine the patient lying **face-up, feet-first** in the scanner:

```
                   Superior (+Z)
                       |
                       |  (toward head)
                       |
    Right (-X) --------+-------- Left (+X)
                      /|
                     / |
           Anterior /  |
             (-Y)      |
                   Posterior (+Y)
```

- **+X** = patient's **L**eft
- **+Y** = patient's **P**osterior (toward the back)
- **+Z** = **S**uperior (toward the head)

### 4.2 RAS — NIfTI Standard

```
                   Superior (+Z)
                       |
                       |  (toward head)
                       |
    Left (-X) ---------+-------- Right (+X)
                      /|
                     / |
          Posterior /   |
             (-Y)      |
                   Anterior (+Y)
```

- **+X** = patient's **R**ight
- **+Y** = patient's **A**nterior (toward the front)
- **+Z** = **S**uperior (toward the head)

### 4.3 Converting Between LPS and RAS

The Z-axis is the same. X and Y are **flipped**:

$$\begin{bmatrix} x_{RAS} \\ y_{RAS} \\ z_{RAS} \end{bmatrix} = \begin{bmatrix} -1 & 0 & 0 \\ 0 & -1 & 0 \\ 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} x_{LPS} \\ y_{LPS} \\ z_{LPS} \end{bmatrix}$$

In code: `ras_point = [-lps_x, -lps_y, lps_z]`

### 4.4 Why Does This Matter?

If you ignore coordinate systems:
- Your PET and CT won't align (left-right flips are common bugs)
- Segmentation masks will be mirrored
- Registration will fail silently
- You'll train on corrupted data and wonder why your model doesn't work

**Rule of thumb:** Always check orientation after loading. Use NiBabel's `nib.aff2axcodes(affine)` to verify.

---
## 5. Affine Matrices — The Key to Everything

The **4x4 affine matrix** is the single most important concept for working with medical images correctly. It maps **voxel indices** (i, j, k) to **world coordinates** (x, y, z) in millimeters.

### The transformation

$$\begin{bmatrix} x \\ y \\ z \\ 1 \end{bmatrix} = \mathbf{M} \cdot \begin{bmatrix} i \\ j \\ k \\ 1 \end{bmatrix}$$

Where **M** is the 4x4 affine matrix:

$$\mathbf{M} = \begin{bmatrix} r_{00} \cdot s_x & r_{01} \cdot s_y & r_{02} \cdot s_z & t_x \\ r_{10} \cdot s_x & r_{11} \cdot s_y & r_{12} \cdot s_z & t_y \\ r_{20} \cdot s_x & r_{21} \cdot s_y & r_{22} \cdot s_z & t_z \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

### Components

| Columns 0-2 (upper-left 3x3) | Column 3 (last column) |
|---|---|
| **Rotation** (direction cosines) x **Scaling** (voxel size) | **Translation** (origin in world coordinates) |
| Encodes: orientation + voxel dimensions | Encodes: where voxel (0,0,0) sits in the world |

### Building from DICOM Tags

1. **`ImageOrientationPatient`** gives you 6 direction cosines: row direction (3) + column direction (3)
2. **`PixelSpacing`** gives you row and column spacing in mm
3. **Slice direction** is computed from `ImagePositionPatient` of the first and last slices
4. **`ImagePositionPatient`** of the first slice gives you the origin

The affine matrix encapsulates ALL of this geometry in one elegant structure.

In [ ]:
# =============================================================================
# Cell 12: Affine Matrix Construction and Usage
# =============================================================================
# Here we build an affine matrix from DICOM tags and demonstrate
# voxel <-> world coordinate transformations.

def build_affine_from_dicom(sorted_slices):
    """
    Build a 4x4 affine matrix from DICOM metadata (LPS convention).
    
    This is the same math that dcm2niix, SimpleITK, and other converters use.
    
    Parameters
    ----------
    sorted_slices : list of pydicom.Dataset
        DICOM slices sorted by SliceLocation.
    
    Returns
    -------
    affine : np.ndarray, shape (4, 4)
        Affine matrix in LPS convention.
    """
    ds = sorted_slices[0]
    
    # Direction cosines from ImageOrientationPatient
    # First 3: direction of rows (column index increases along this direction)
    # Last 3: direction of columns (row index increases along this direction)
    iop = [float(x) for x in ds.ImageOrientationPatient]
    row_cosine = np.array(iop[:3])  # Direction along columns (i-axis)
    col_cosine = np.array(iop[3:])  # Direction along rows (j-axis)
    
    # Pixel spacing: [row_spacing, col_spacing] in mm
    ps = [float(x) for x in ds.PixelSpacing]
    
    # Slice direction: computed from positions of first and last slices
    ipp_first = np.array([float(x) for x in sorted_slices[0].ImagePositionPatient])
    ipp_last = np.array([float(x) for x in sorted_slices[-1].ImagePositionPatient])
    n = len(sorted_slices)
    
    if n > 1:
        # The slice direction vector, including spacing
        slice_dir = (ipp_last - ipp_first) / (n - 1)
    else:
        # Fallback: cross product of row and column cosines, scaled by slice thickness
        slice_dir = np.cross(row_cosine, col_cosine) * float(ds.SliceThickness)
    
    # Build 4x4 affine matrix
    # Column 0: how world coordinates change when column index (i) increases by 1
    # Column 1: how world coordinates change when row index (j) increases by 1
    # Column 2: how world coordinates change when slice index (k) increases by 1
    # Column 3: world coordinates of voxel (0, 0, 0)
    affine = np.eye(4)
    affine[:3, 0] = row_cosine * ps[1]    # Column direction, scaled by column spacing
    affine[:3, 1] = col_cosine * ps[0]    # Row direction, scaled by row spacing
    affine[:3, 2] = slice_dir             # Slice direction (already includes spacing)
    affine[:3, 3] = ipp_first             # Origin = position of first voxel
    
    return affine


# --- Build and inspect the affine ---
affine_lps = build_affine_from_dicom(sorted_slices)

print("Affine matrix (LPS convention):")
print(np.array2string(affine_lps, precision=2, suppress_small=True))

print("\n--- Voxel -> World coordinate examples ---")
test_voxels = [
    (0, 0, 0, 1),
    (32, 32, 16, 1),
    (63, 63, 31, 1),
]
for v in test_voxels:
    world = affine_lps @ np.array(v)
    print(f"  Voxel {v[:3]} -> World (LPS): ({world[0]:.1f}, {world[1]:.1f}, {world[2]:.1f}) mm")

print("\n--- World -> Voxel (inverse transform) ---")
inv_affine = np.linalg.inv(affine_lps)
# Round-trip: voxel -> world -> voxel should give back the same indices
world_point = affine_lps @ np.array([32, 32, 16, 1])
voxel_back = inv_affine @ world_point
print(f"  World ({world_point[0]:.1f}, {world_point[1]:.1f}, {world_point[2]:.1f}) -> Voxel: ({voxel_back[0]:.1f}, {voxel_back[1]:.1f}, {voxel_back[2]:.1f})")

# Verify round-trip accuracy
assert np.allclose(voxel_back[:3], [32, 32, 16]), "Round-trip failed!"
print("  Round-trip verified!")

# --- Decompose the affine for insight ---
print("\n--- Affine decomposition ---")
print(f"  Voxel sizes: {np.linalg.norm(affine_lps[:3, 0]):.2f} x {np.linalg.norm(affine_lps[:3, 1]):.2f} x {np.linalg.norm(affine_lps[:3, 2]):.2f} mm")
print(f"  Origin (LPS): ({affine_lps[0, 3]:.1f}, {affine_lps[1, 3]:.1f}, {affine_lps[2, 3]:.1f}) mm")

# Check orientation using NiBabel
print(f"  Axis codes: {nib.aff2axcodes(affine_lps)}")
print("  (Should show L,P,S for standard DICOM axial orientation)")

In [ ]:
# =============================================================================
# Cell 13: DICOM to NIfTI Conversion
# =============================================================================
# This is one of the most common operations in medical imaging research.
# The key subtlety: DICOM uses LPS, NIfTI uses RAS. We must flip X and Y.

def dicom_to_nifti(sorted_slices, output_path):
    """
    Convert a sorted DICOM series to NIfTI format.
    
    This handles:
    1. Building the 3D volume from 2D slices
    2. Applying RescaleSlope/Intercept
    3. Converting the affine from LPS (DICOM) to RAS (NIfTI)
    
    Parameters
    ----------
    sorted_slices : list of pydicom.Dataset
        DICOM slices sorted by SliceLocation.
    output_path : str
        Path to save the NIfTI file (.nii or .nii.gz).
    
    Returns
    -------
    img : nib.Nifti1Image
        The saved NIfTI image.
    """
    # Step 1: Build 3D volume from 2D slices
    volume = np.stack(
        [s.pixel_array.astype(np.float32) for s in sorted_slices],
        axis=0
    )
    
    # Step 2: Apply rescale (stored_value * slope + intercept = real_value)
    if hasattr(sorted_slices[0], 'RescaleSlope'):
        slope = float(sorted_slices[0].RescaleSlope)
        intercept = float(sorted_slices[0].RescaleIntercept)
        volume = volume * slope + intercept
    
    # Step 3: Build LPS affine from DICOM geometry
    lps_affine = build_affine_from_dicom(sorted_slices)
    
    # Step 4: Convert LPS -> RAS
    # Negate X and Y axes (Left->Right, Posterior->Anterior)
    lps_to_ras = np.diag([-1, -1, 1, 1])
    ras_affine = lps_to_ras @ lps_affine
    
    # Step 5: Save as NIfTI
    img = nib.Nifti1Image(volume, ras_affine)
    nib.save(img, output_path)
    
    print(f"Saved NIfTI: {output_path}")
    print(f"  Volume shape: {volume.shape}")
    print(f"  Value range:  [{volume.min():.1f}, {volume.max():.1f}]")
    print(f"  LPS affine origin: ({lps_affine[0,3]:.1f}, {lps_affine[1,3]:.1f}, {lps_affine[2,3]:.1f})")
    print(f"  RAS affine origin: ({ras_affine[0,3]:.1f}, {ras_affine[1,3]:.1f}, {ras_affine[2,3]:.1f})")
    print(f"  Axis codes: {nib.aff2axcodes(ras_affine)}")
    
    return img


# --- Perform the conversion ---
nifti_pet_path = os.path.join(tmpdir, 'synthetic_pet.nii.gz')
converted_img = dicom_to_nifti(sorted_slices, nifti_pet_path)

# --- Verify the conversion ---
print("\n" + "=" * 60)
print("VERIFICATION: Compare DICOM volume vs NIfTI volume")
print("=" * 60)

reloaded = nib.load(nifti_pet_path)
reloaded_data = reloaded.get_fdata()

print(f"  DICOM volume shape: {volume_from_dicom.shape}")
print(f"  NIfTI volume shape: {reloaded_data.shape}")

# The data should be identical (modulo floating-point precision)
max_diff = np.abs(volume_from_dicom - reloaded_data).max()
print(f"  Max voxel difference: {max_diff:.6f}")
print(f"  Volumes match: {max_diff < 1e-3}")

# Show both affines side by side
print(f"\n  LPS Affine (DICOM):")
print(f"  {np.array2string(build_affine_from_dicom(sorted_slices), precision=1)}")
print(f"\n  RAS Affine (NIfTI):")
print(f"  {np.array2string(reloaded.affine, precision=1)}")

In [ ]:
# =============================================================================
# Cell 14: Interactive Slice Viewer
# =============================================================================
# If ipywidgets is available, this creates a slider-based viewer.
# Otherwise, falls back to a static multi-slice display.

if HAS_WIDGETS:
    # --- Interactive version with ipywidgets ---
    
    def interactive_viewer(volume, voxel_spacing=(4.0, 4.0, 4.0),
                           cmap='hot', title='Interactive Slice Viewer'):
        """
        Create an interactive 3-plane viewer with sliders for each axis.
        """
        nz, ny, nx = volume.shape
        sz, sy, sx = voxel_spacing
        vmax = np.percentile(volume, 99.5)
        vmin = np.percentile(volume, 0.5)
        
        def _update(axial=nz//2, coronal=ny//2, sagittal=nx//2):
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            fig.suptitle(title, fontsize=14, fontweight='bold')
            
            # Axial
            axes[0].imshow(volume[axial], cmap=cmap, vmin=vmin, vmax=vmax)
            axes[0].axhline(y=coronal, color='cyan', alpha=0.5, lw=0.8)
            axes[0].axvline(x=sagittal, color='cyan', alpha=0.5, lw=0.8)
            axes[0].set_title(f'Axial (z={axial})')
            
            # Coronal
            axes[1].imshow(volume[:, coronal, :], cmap=cmap, vmin=vmin, vmax=vmax,
                          aspect=sz/sx)
            axes[1].axhline(y=axial, color='cyan', alpha=0.5, lw=0.8)
            axes[1].axvline(x=sagittal, color='cyan', alpha=0.5, lw=0.8)
            axes[1].set_title(f'Coronal (y={coronal})')
            
            # Sagittal
            axes[2].imshow(volume[:, :, sagittal], cmap=cmap, vmin=vmin, vmax=vmax,
                          aspect=sz/sy)
            axes[2].axhline(y=axial, color='cyan', alpha=0.5, lw=0.8)
            axes[2].axvline(x=coronal, color='cyan', alpha=0.5, lw=0.8)
            axes[2].set_title(f'Sagittal (x={sagittal})')
            
            plt.tight_layout()
            plt.show()
        
        interact(
            _update,
            axial=IntSlider(min=0, max=nz-1, value=nz//2, description='Axial (z):'),
            coronal=IntSlider(min=0, max=ny-1, value=ny//2, description='Coronal (y):'),
            sagittal=IntSlider(min=0, max=nx-1, value=nx//2, description='Sagittal (x):'),
        )
    
    print("Interactive viewer loaded. Use the sliders to navigate the volume.")
    interactive_viewer(volume_from_dicom, title='Synthetic PET — Interactive Viewer')

else:
    # --- Static fallback ---
    print("ipywidgets not available. Showing static multi-slice view.\n")
    
    # Show 5 evenly-spaced axial slices
    n_show = 5
    indices = np.linspace(0, volume_from_dicom.shape[0] - 1, n_show, dtype=int)
    vmax = np.percentile(volume_from_dicom, 99.5)
    
    fig, axes = plt.subplots(1, n_show, figsize=(16, 4))
    fig.suptitle('Synthetic PET — Selected Axial Slices', fontsize=14, fontweight='bold')
    for ax, idx in zip(axes, indices):
        ax.imshow(volume_from_dicom[idx], cmap='hot', vmin=0, vmax=vmax)
        ax.set_title(f'z={idx}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

---
## 6. Package Ecosystem Overview

Medical imaging has a rich ecosystem of specialized tools. Here are the ones you'll encounter most often:

### SimpleITK
**The Swiss army knife of medical image processing.**
- Wraps the [Insight Toolkit (ITK)](https://itk.org/) in a clean Python API
- Handles: reading/writing all formats, resampling, registration, filtering, segmentation
- Correctly manages metadata and coordinate systems throughout all operations
- `pip install SimpleITK`

### TotalSegmentator
**One-line CT organ/structure segmentation.**
- Pre-trained nnU-Net model that segments 117 anatomical structures from CT
- Usage: `TotalSegmentator -i ct.nii.gz -o segmentations/`
- Produces individual NIfTI masks for each structure (liver, kidneys, vertebrae, etc.)
- Extremely useful for creating training labels or anatomical priors
- `pip install TotalSegmentator`

### MONAI
**PyTorch framework purpose-built for medical imaging deep learning.**
- Medical-specific transforms (spatial: resample, crop, augment; intensity: normalize, window)
- Dataset classes that handle NIfTI, DICOM, and caching
- Pre-built architectures: UNet, SwinUNETR, SegResNet, ViT
- Loss functions: DiceLoss, DiceCELoss, FocalLoss
- Metrics: DiceMetric, HausdorffDistance
- `pip install monai`

### 3D Slicer
**Open-source GUI for visualization and annotation.**
- Load and overlay multiple volumes (PET on CT, segmentation masks)
- Manual segmentation tools (paint, threshold, grow from seeds)
- Extensible via Python scripting
- Essential for QA: always visually inspect your data pipeline output
- Download: [slicer.org](https://www.slicer.org/)

In [ ]:
# =============================================================================
# Cell 16: SimpleITK Demo
# =============================================================================
# SimpleITK provides a higher-level API than raw nibabel/pydicom.
# It manages coordinate systems automatically and provides powerful
# image processing operations.

# --- Load NIfTI with SimpleITK ---
sitk_img = sitk.ReadImage(nifti_pet_path)

print("SimpleITK Image Properties:")
print(f"  Size:      {sitk_img.GetSize()}")          # (x, y, z) — note: reversed from numpy!
print(f"  Spacing:   {sitk_img.GetSpacing()} mm")    # Voxel size
print(f"  Origin:    {sitk_img.GetOrigin()} mm")     # World coordinates of voxel (0,0,0)
print(f"  Direction: {sitk_img.GetDirection()}")      # 9 direction cosines (flattened 3x3)
print(f"  Pixel type: {sitk_img.GetPixelIDTypeAsString()}")

# --- Convert to numpy for comparison ---
sitk_array = sitk.GetArrayFromImage(sitk_img)  # Returns (z, y, x) order!
print(f"\n  NumPy array shape: {sitk_array.shape}")
print(f"  Note: SimpleITK uses (x,y,z) but GetArrayFromImage returns (z,y,x)!")

# --- Demo: Gaussian Smoothing ---
# Smoothing is commonly used for denoising or as a preprocessing step.
# Sigma is specified in physical units (mm), NOT voxels!
smoothed = sitk.SmoothingRecursiveGaussian(sitk_img, sigma=8.0)  # 8mm FWHM
smoothed_array = sitk.GetArrayFromImage(smoothed)

print(f"\n--- Gaussian Smoothing (sigma=8mm) ---")
print(f"  Original range: [{sitk_array.min():.1f}, {sitk_array.max():.1f}]")
print(f"  Smoothed range: [{smoothed_array.min():.1f}, {smoothed_array.max():.1f}]")

# --- Demo: Resampling to different resolution ---
# Resampling changes voxel size while preserving physical extent.
# This is crucial for standardizing inputs to neural networks.
new_spacing = [2.0, 2.0, 2.0]  # Upsample from 4mm to 2mm

# Compute new size to maintain physical coverage
original_size = sitk_img.GetSize()
original_spacing = sitk_img.GetSpacing()
new_size = [
    int(round(osz * ospc / nspc))
    for osz, ospc, nspc in zip(original_size, original_spacing, new_spacing)
]

resampler = sitk.ResampleImageFilter()
resampler.SetOutputSpacing(new_spacing)
resampler.SetSize(new_size)
resampler.SetOutputDirection(sitk_img.GetDirection())
resampler.SetOutputOrigin(sitk_img.GetOrigin())
resampler.SetInterpolator(sitk.sitkLinear)  # Use BSpline for CT, Linear for PET/labels
resampled = resampler.Execute(sitk_img)

print(f"\n--- Resampling (4mm -> 2mm isotropic) ---")
print(f"  Original: size={original_size}, spacing={original_spacing}")
print(f"  Resampled: size={resampled.GetSize()}, spacing={resampled.GetSpacing()}")

# --- Visualize original vs smoothed vs resampled ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('SimpleITK Processing Demo', fontsize=14, fontweight='bold')

mid_z = sitk_array.shape[0] // 2
vmax = np.percentile(sitk_array, 99.5)

axes[0].imshow(sitk_array[mid_z], cmap='hot', vmin=0, vmax=vmax)
axes[0].set_title(f'Original (4mm, {sitk_array.shape[1]}x{sitk_array.shape[2]})')
axes[0].axis('off')

axes[1].imshow(smoothed_array[mid_z], cmap='hot', vmin=0, vmax=vmax)
axes[1].set_title(f'Gaussian Smoothed (sigma=8mm)')
axes[1].axis('off')

resampled_array = sitk.GetArrayFromImage(resampled)
mid_z_r = resampled_array.shape[0] // 2
axes[2].imshow(resampled_array[mid_z_r], cmap='hot', vmin=0, vmax=vmax)
axes[2].set_title(f'Resampled (2mm, {resampled_array.shape[1]}x{resampled_array.shape[2]})')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("\nKey takeaway: SimpleITK operates in physical coordinates.")
print("Smoothing sigma and resampling spacing are in millimeters, not pixels.")
print("This makes your code resolution-independent.")

---
## 7. Summary and Next Steps

### What We Covered

1. **Physics primer**: PET measures metabolic activity (SUV), SPECT measures perfusion, CT measures density (HU). PET/SPECT = function, CT = anatomy.

2. **DICOM format**: One file per slice, rich metadata. Key tags: `PixelSpacing`, `ImageOrientationPatient`, `ImagePositionPatient`, `RescaleSlope/Intercept`.

3. **NIfTI format**: Single file per volume, minimal metadata, research standard. Header + affine + data array.

4. **Coordinate systems**: DICOM uses LPS, NIfTI uses RAS. Converting between them means negating X and Y. Always verify orientation after loading.

5. **Affine matrices**: The 4x4 matrix maps voxel indices to world coordinates. Built from direction cosines, pixel spacing, and image position. This is the foundation of all multi-modal registration.

6. **DICOM to NIfTI conversion**: Build volume from slices, apply rescale, convert LPS affine to RAS, save with NiBabel.

7. **SimpleITK**: High-level API for medical image processing that correctly manages coordinates, supports resampling, filtering, and registration.

### Key Takeaways for ML Engineers

- **Never ignore metadata.** Voxel spacing, orientation, and coordinate systems are not optional.
- **Always resample to a standard resolution** before feeding data into a neural network.
- **Verify orientation visually** using orthogonal views. Left-right flips are a common silent bug.
- **Use NIfTI for your ML pipeline** — convert from DICOM early, and keep the conversion reproducible.

### Next Notebooks

- **Notebook II: Image Reconstruction** — How raw detector data becomes the 3D volumes you just learned to load. Covers sinograms, filtered back-projection, iterative reconstruction (OSEM), and attenuation/scatter correction.
- **Notebook III: Preprocessing & Augmentation** — Windowing, normalization, spatial transforms, and MONAI data pipelines for training.

---
*Notebook by Xingfu Yang. Part of the Deep Learning for Medical Imaging tutorial series.*